# Pipeline de ML

Quero prever o total geral futuro considerando apenas rotas reais (já vendidas), preservando a distribuição original dos dados, de modo que rotas raras contribuam para a previsão exatamente na proporção em que aparecem no histórico, sem criação de novas conexões origem–destino e sem reponderação artificial.

## 1. Instalação e importação de bibliotecas
Ela será usada para remover acentos de textos (ex: “São Paulo” → “Sao Paulo”), evitando duplicações artificiais de categorias.

In [ ]:
!pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 1.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

from unidecode import unidecode
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

A biblioteca ``unidecode`` não vem instalada por padrão no Colab, por isso precisa ser instalada antes de qualquer coisa. Ela será usada para normalizar textos com acentos e caracteres especiais (por exemplo, converter "São Paulo" em "SAO PAULO") evitando que o modelo trate variações do mesmo nome como categorias distintas.

As demais bibliotecas importadas são:

``pandas``: manipulação de dados em formato tabular (DataFrames). É a principal ferramenta de preparação dos dados ao longo de todo o notebook.

``numpy``: operações numéricas de baixo nível, usada aqui principalmente para cálculo de métricas (raiz quadrada do erro quadrático médio).

``unidecode``: normalização de texto, removendo acentos e caracteres especiais.

``LabelEncoder`` (sklearn): converte variáveis categóricas (textos como nomes de cidades e operadoras) em números inteiros, que é o formato que modelos de Machine Learning conseguem processar.

``RandomForestRegressor`` (sklearn): o modelo preditivo que será usado. É um algoritmo baseado em múltiplas árvores de decisão treinadas em paralelo; mais detalhes na seção de treinamento.

``mean_absolute_error``, ``mean_squared_error``, ``r2_score`` (sklearn): métricas para avaliar a **qualidade** das previsões do modelo.

## 2. Leitura dos dados

In [ ]:
# Ler CSV
#df = pd.read_csv( "relatorio_rotas-1-25-nov-para-colab.csv", sep=";", decimal=",")
df = pd.read_csv( "diario_2026-01-01_2026-05-17.csv", decimal=",")
df.head()

,dia,mes,ano,operadora,origem,destino,canal_de_venda,bilhetes,ordens,receita,gmv
0,1,1,2026,1001,IGUABA GRANDE - RJ,RIO DE JANEIRO - NOVO RIO - RJ,App,4,1,35.3872,243.5472
1,1,1,2026,Andorinha,CAMPINAS - SP,PRESIDENTE PRUDENTE - SP,App,2,1,82.2282,786.7782
2,1,1,2026,Andorinha,TRES LAGOAS - MS,CAMPO GRANDE - MS,Web,1,1,22.6400,158.3100
3,1,1,2026,Bragança,SÃO PAULO - RODOVIÁRIA TIETÊ - SP,BRAGANCA PAULISTA - SP,App,1,1,8.2450,56.7450
4,1,1,2026,Brasil Sul,ARARANGUA - SC,PORTO ALEGRE - RS,Web,2,1,66.2086,289.7886


Os dados são carregados a partir de um arquivo CSV exportado do sistema de vendas. Dois parâmetros importantes na leitura:

``sep=";"``: o arquivo usa ponto-e-vírgula como separador de colunas, ao invés da vírgula padrão. Isso é comum em exportações de sistemas brasileiros.


``decimal=","``: os valores decimais estão no formato brasileiro (vírgula como separador decimal), então o pandas precisa saber interpretar "1.234,56" corretamente.

Cada linha do dataset representa um agregado diário de vendas para uma combinação específica de data, rota e operadora.

>Ou seja, se em um mesmo dia foram feitos 2 pedidos para a rota Curitiba–Ponta Grossa pela mesma operadora (um pedido com 2 bilhetes e outro com 3 bilhetes) esses dados aparecem consolidados em uma única linha, com ordens = 2 e bilhetes = 5.

## 3. Normalização de texto (operadora, origem, destino)

Função criada para padronizar textos categóricos.

In [ ]:
def normalizar_texto(x):
    if isinstance(x, str):
        x = x.strip()           # tira espaços na ponta
        x = x.replace("_", " ") # troca _ por espaço
        x = " ".join(x.split()) # remove espaços duplicados
        x = unidecode(x)        # remove acentos
        x = x.upper()           # tudo maiúsculo
        return x
    return x

for col in ["operadora", "origem", "destino"]:
    df[col] = df[col].apply(normalizar_texto)

df[["operadora", "origem", "destino"]].head(10)

,operadora,origem,destino
0,1001,IGUABA GRANDE - RJ,RIO DE JANEIRO - NOVO RIO - RJ
1,ANDORINHA,CAMPINAS - SP,PRESIDENTE PRUDENTE - SP
2,ANDORINHA,TRES LAGOAS - MS,CAMPO GRANDE - MS
3,BRAGANCA,SAO PAULO - RODOVIARIA TIETE - SP,BRAGANCA PAULISTA - SP
4,BRASIL SUL,ARARANGUA - SC,PORTO ALEGRE - RS
5,BRASIL SUL,BALNEARIO CAMBORIU - SC,CURITIBA - PR
6,BRASIL SUL,CAXIAS DO SUL - RS,CURITIBA - PR
7,BRASIL SUL,CRICIUMA - SC,PORTO ALEGRE - RS
8,BRASIL SUL,CRICIUMA - SC,PORTO ALEGRE - RS
9,BRASIL SUL,CURITIBA - PR,CAXIAS DO SUL - RS


Antes de qualquer transformação, é necessário padronizar os textos das colunas categóricas (operadora, origem, destino). Isso porque sistemas de vendas frequentemente registram o mesmo valor de formas ligeiramente diferentes ao longo do tempo, seja por digitação manual, exportações distintas ou atualizações de cadastro. Exemplos comuns:

>``"São Paulo"``, ``"são_paulo"``, ``"SAO PAULO"``, ``"São Paulo "`` (com espaço no final).

Para um modelo de Machine Learning, essas variações seriam tratadas como categorias completamente diferentes, o que distorceria o aprendizado. A função ``normalizar_texto`` resolve isso aplicando cinco transformações em sequência:

``.strip()``: remove espaços em branco no início e no fim da string. Um espaço invisível no final faria "Curitiba" e "Curitiba " serem tratados como cidades diferentes.

``.replace("_", " ")``: substitui underscores por espaços. Exportações de sistemas às vezes gravam "Ponta_Grossa" em vez de "Ponta Grossa".

``" ".join(x.split())``: remove espaços duplicados internos. "Ponta  Grossa" (dois espaços) vira "Ponta Grossa".

``unidecode(x)``: remove acentos e caracteres especiais. "São Paulo" vira "Sao Paulo", "Goiânia" vira "Goiania". Isso evita problemas de encoding entre diferentes ambientes.

``.upper()``: converte tudo para maiúsculas. Garante que "curitiba", "Curitiba" e "CURITIBA" sejam tratados como a mesma categoria.

A função é então aplicada nas três colunas de texto via ``.apply()``, que percorre cada célula da coluna e aplica a transformação. O ``if isinstance(x, str)`` garante que valores nulos ``(NaN)`` ou numéricos não quebrem a função — nesses casos, o valor é retornado sem modificação.

In [ ]:
df.dtypes

,0
dia,int64
mes,int64
ano,int64
operadora,object
origem,object
destino,object
canal_de_venda,object
bilhetes,int64
ordens,int64
receita,float64


## 4. Criação da coluna de data e ordenação temporal

In [ ]:
# Coluna de data
df["data"] = pd.to_datetime(dict(year=df["ano"], month=df["mes"], day=df["dia"])) # Constrói uma data real (datetime) a partir de colunas separadas

# Ordenar temporalmente
df = df.sort_values(by="data").reset_index(drop=True)
# Ordena o dataset cronologicamente, o que é essencial para: previsões temporais; split treino/teste sem vazamento de informação

O dataset original armazena a data em três colunas separadas: ano, mês e dia. Isso é comum em exportações de sistemas legados, mas impede qualquer operação temporal direta (ordenação cronológica, extração de dia da semana, cálculo de diferença entre datas etc.).

O ``pd.to_datetime(dict(...))`` constrói uma data real no formato datetime a partir dessas três colunas separadas, unificando-as em uma única coluna data. O resultado é uma coluna que o pandas reconhece como data de verdade, permitindo, por exemplo, descobrir que 2025-11-22 é uma sábado com um simples .dt.weekday.

Em seguida, o dataset é **ordenado cronologicamente** com ``.sort_values(by="data")``. Essa etapa é essencial em qualquer modelo de previsão temporal por dois motivos:

*   Divisão treino/teste sem vazamento de informação: quando dividirmos os dados em treino e teste mais adiante, precisamos garantir que o modelo aprenda **apenas com o passado e seja testado no futuro, nunca o contrário**. Se os dados estivessem embaralhados, dias de dezembro poderiam aparecer no treino e dias de janeiro no teste, o que tornaria a avaliação do modelo completamente inválida.

*   Integridade das features temporais: algumas features que serão criadas a seguir (como dia_semana e semana_ano) só fazem sentido se os dados estiverem em ordem cronológica correta.

O ``.reset_index(drop=True)`` ao final apenas reorganiza os índices do DataFrame após a ordenação, evitando índices fora de sequência que poderiam causar problemas em operações futuras.



## 5. Criação da rota e features de tempo

A coluna rota é criada concatenando origem e destino com um separador -. O resultado é uma string como "CURITIBA - PONTA GROSSA" que representa o par origem–destino de forma única.

Por que criar essa coluna se já temos origem e destino separadas? Porque capturar o par completo como uma única feature permite ao modelo aprender padrões específicos de cada rota, por exemplo, que "CURITIBA - PONTA GROSSA" tem um volume de vendas muito diferente de "CURITIBA - SAO PAULO", mesmo que a origem seja a mesma. Se usássemos apenas origem e destino separadamente, o modelo teria dificuldade de capturar essa interação.

In [ ]:
# Rota normalizada
df["rota"] = df["origem"] + " - " + df["destino"] # Cria uma feature de rota completa, capturando o par origem–destino.

Features de tempo são variáveis derivadas da data que ajudam o modelo a aprender padrões sazonais e cíclicos de vendas. Quatro features são criadas:


*   dia_semana: número inteiro de 0 (segunda-feira) a 6 (domingo). Vendas de transporte tendem a ter padrões muito distintos entre dias úteis e fins de semana: uma segunda-feira chuvosa de novembro e um sábado de novembro têm perfis de demanda completamente diferentes, mesmo sendo do mesmo mês.

*   semana_ano: número da semana dentro do ano (1 a 52/53), extraído via .``isocalendar()``. Captura sazonalidade de médio prazo: por exemplo, semanas de feriados prolongados ou de férias escolares tendem a ter volume de vendas diferente das demais.

*   mes_num: número do mês (1 a 12), aproveitado diretamente da coluna mês já existente. Captura sazonalidade anual: meses de férias (janeiro, julho) e períodos de alta demanda tendem a se repetir ano a ano.

*   fim_semana: flag binária (0 ou 1) que indica se o dia é sábado ou domingo. ``.isin([5, 6])`` retorna True para esses dias, e ``.astype(int)`` converte para 0/1. Essa feature é uma simplificação de dia_semana que destaca explicitamente o comportamento diferenciado dos fins de semana, que em transporte rodoviário costuma ser um dos sinais mais fortes.

In [ ]:
# Features de tempo
df["dia_semana"] = df["data"].dt.weekday
df["semana_ano"] = df["data"].dt.isocalendar().week.astype(int)
df["mes_num"]    = df["mes"]
df["fim_semana"] = df["dia_semana"].isin([5, 6]).astype(int)

df.head()

# Cria features temporais que ajudam o modelo a aprender padrões: dia_semana > segunda(0) até domingo(6); semana_ano > sazonalidade semanal; mes_num > sazonalidade mensal; fim_semana > flag binária (0/1)

,dia,mes,ano,operadora,origem,destino,canal_de_venda,bilhetes,ordens,receita,gmv,data,rota,dia_semana,semana_ano,mes_num,fim_semana
0,1,1,2026,1001,IGUABA GRANDE - RJ,RIO DE JANEIRO - NOVO RIO - RJ,App,4,1,35.3872,243.5472,2026-01-01,IGUABA GRANDE - RJ - RIO DE JANEIRO - NOVO RIO...,3,1,1,0
1,1,1,2026,PRINCESA DOS CAMPOS,PONTA GROSSA - PR,SAO MIGUEL DO OESTE - SC,App,1,1,21.1600,239.8500,2026-01-01,PONTA GROSSA - PR - SAO MIGUEL DO OESTE - SC,3,1,1,0
2,1,1,2026,PRINCESA DOS CAMPOS,PONTA GROSSA - PR,TELEMACO BORBA - PR,Web,1,1,18.3334,85.6134,2026-01-01,PONTA GROSSA - PR - TELEMACO BORBA - PR,3,1,1,0
3,1,1,2026,PRINCESA DOS CAMPOS,PONTA GROSSA - PR,TIBAGI - PR,Web,1,1,9.6101,53.2701,2026-01-01,PONTA GROSSA - PR - TIBAGI - PR,3,1,1,0
4,1,1,2026,PRINCESA DOS CAMPOS,PRUDENTOPOLIS - PR,PONTA GROSSA - PR,Web,1,1,10.0738,56.3238,2026-01-01,PRUDENTOPOLIS - PR - PONTA GROSSA - PR,3,1,1,0


Em conjunto, essas quatro features formam um "calendário codificado" que o modelo pode usar para aprender que, por exemplo, sextas-feiras da semana 47 de novembro tendem a ter mais vendas do que terças-feiras de fevereiro; sem precisar ver datas futuras, apenas os padrões do passado.

## 6. Limpeza de colunas antigas

In [ ]:
df = df.drop(columns=["operadora_encoded", "origem_encoded", "destino_encoded"], errors="ignore")
# Remove colunas antigas caso existam, evitando conflitos; errors="ignore" impede erro se elas não existirem.

Essa etapa remove colunas de encoding que possam ter sido geradas em execuções anteriores do notebook. O parâmetro ``errors="ignore"`` garante que, se essas colunas não existirem, o código simplesmente não faz nada, sem gerar erro.

Por que isso é necessário? Notebooks no Colab mantêm o estado das variáveis em memória enquanto a sessão está ativa. Se rodar o notebook duas vezes sem reiniciar o ambiente, colunas criadas na primeira execução ainda existem no DataFrame da segunda execução. Sem essa limpeza, o Label Encoding da próxima etapa poderia gerar colunas duplicadas ou conflitantes (como ``operadora_encoded`` e ``operadora_enc`` coexistindo com valores diferentes), o que quebraria o modelo silenciosamente, sem erro aparente, mas com resultados incorretos.

É uma boa prática incluir esse tipo de limpeza defensiva sempre que um notebook pode ser reexecutado parcialmente.

## 7. Label encoding

Modelos de Machine Learning não conseguem processar texto diretamente, eles operam exclusivamente com números. O **Label Encoding** é uma das técnicas para fazer essa conversão: cada categoria única recebe um número inteiro arbitrário.
Por exemplo, se as operadoras no dataset forem "GARCIA", "CATARINENSE" e "REUNIDAS", o LabelEncoder pode atribuir:



"CATARINENSE" >>> 0

"GARCIA" >>> 1

"REUNIDAS" >>> 2

A ordem é alfabética, não tem significado numérico real, o modelo aprende isso durante o treinamento.

In [ ]:
le_operadora = LabelEncoder()
le_origem    = LabelEncoder()
le_destino   = LabelEncoder()
le_rota      = LabelEncoder()

df["operadora_enc"] = le_operadora.fit_transform(df["operadora"])
df["origem_enc"]    = le_origem.fit_transform(df["origem"])
df["destino_enc"]   = le_destino.fit_transform(df["destino"])
df["rota_enc"]      = le_rota.fit_transform(df["rota"])

df[["operadora", "operadora_enc", "origem", "origem_enc"]].head()

,operadora,operadora_enc,origem,origem_enc
0,1001,0,IGUABA GRANDE - RJ,315
1,PRINCESA DOS CAMPOS,113,PONTA GROSSA - PR,608
2,PRINCESA DOS CAMPOS,113,PONTA GROSSA - PR,608
3,PRINCESA DOS CAMPOS,113,PONTA GROSSA - PR,608
4,PRINCESA DOS CAMPOS,113,PRUDENTOPOLIS - PR,637


**Por que um encoder separado para cada coluna?**

Cada LabelEncoder é um objeto independente que "memoriza" o mapeamento **categoria >>> número** que ele aprendeu. Ter encoders separados é fundamental por dois motivos:



1.   *Evitar colisão de índices*: se usássemos um único encoder para todas as colunas, "CURITIBA" (origem) e "CURITIBA" (destino) receberiam o mesmo número (o que até faz sentido) mas "GARCIA" (operadora) poderia receber o mesmo número que "FLORIANOPOLIS" (cidade), criando uma ambiguidade que confunde o modelo.

2.   *Permitir reuso no futuro*: quando formos construir os dados futuros para previsão, precisaremos transformar as categorias usando exatamente o mesmo mapeamento aprendido no histórico. Como cada encoder é um objeto salvo em memória (le_operadora, le_origem etc.), basta chamar ``.transform()`` neles, sem precisar reaprender o mapeamento.



``.fit_transform() vs .transform()``

Esse é um dos conceitos mais importantes para entender antes de chegar na etapa de previsão:


*   ``.fit_transform()``: usado no histórico. O encoder aprende quais categorias existem (.fit) e já converte para número (.transform) em uma única chamada. É como ensinar o encoder a "traduzir" um idioma e já pedir a tradução ao mesmo tempo.

*   ``.transform()``: usado nos dados futuros. O encoder já sabe o mapeamento (aprendeu no histórico) e apenas aplica a conversão. Se aparecer uma categoria nova que não existia no histórico, ele vai gerar um erro, o que é o comportamento correto, pois o modelo nunca viu aquela categoria e não saberia o que fazer com ela. Por isso existe a etapa de máscara de válidos mais adiante.

**Uma limitação importante do Label Encoding**

O Label Encoding introduz uma ordem numérica implícita que não existe nas categorias originais. Ao codificar "CATARINENSE" >>> 0, "GARCIA" >>> 1 e "REUNIDAS" >>> 2, o modelo pode interpretar que "REUNIDAS" é "maior" ou "mais importante" que "CATARINENSE", **o que não faz sentido**.

Para variáveis com poucas categorias e sem ordem natural, técnicas como **One-Hot Encoding** seriam mais adequadas teoricamente. Porém, no caso de rota, que pode ter centenas de categorias únicas, o One-Hot Encoding criaria centenas de colunas novas, tornando o modelo muito mais pesado e lento. O **Random Forest lida razoavelmente bem com Label Encoding mesmo assim, porque suas árvores de decisão aprendem por divisões binárias**, e acabam descobrindo que "GARCIA" e "REUNIDAS" têm comportamentos distintos independentemente dos números atribuídos.

## 8. Definição de features e target

Antes de treinar qualquer modelo, é necessário separar claramente duas coisas:

*   **Target (variável dependente)**: é o que queremos prever. Aqui é bilhetes, o total de bilhetes vendidos para uma combinação de rota, operadora e dia.

*   **Features (variáveis independentes)**: as informações que o modelo pode usar para fazer a previsão. Tudo que estiver disponível no momento da previsão e que não seja o próprio target.

In [ ]:
# Colunas que NUNCA vão para o modelo
cols_nao_usar = ["bilhetes", "ordens", "receita", "gmv", "operadora", "origem", "destino", "rota", "data", "canal_de_venda"]

# Features = tudo exceto as acima
feature_cols = [c for c in df.columns if c not in cols_nao_usar]

target_col = "bilhetes"

Por que excluir cada coluna da lista ``cols_nao_usar``?


*   bilhetes: é o target; jamais pode ser uma feature, pois no momento da previsão ainda não sabemos esse valor.

*   ordens, receita e gmv: são métricas de resultado, assim como bilhetes. Se o modelo aprendesse que "quando ordens é alto, bilhetes é alto", estaria usando informação que só existe depois que a venda acontece, o que é chamado de **data leakage (vazamento de dados)**. O modelo pareceria ótimo nos testes mas seria inútil na prática, pois não teríamos ordens para alimentá-lo antes de prever.

*   operadora, origem, destino, rota: são as versões em texto das colunas categóricas. Foram substituídas pelas versões numéricas (``operadora_enc``, ``origem_enc``, ``destino_enc``, ``rota_enc``) na etapa anterior. Manter as duas versões causaria redundância.

*   data: a data bruta não é diretamente útil para o modelo, o que importa são os padrões que ela carrega, e esses já foram extraídos nas features de tempo (``dia_semana``, ``semana_ano``, ``mes_num``, ``fim_semana``). Além disso, datas futuras que o modelo nunca viu poderiam confundi-lo se fossem passadas diretamente.

**Data leakage: por que é tão grave?**

Data leakage acontece quando o modelo recebe durante o treino informações que não estariam disponíveis no momento real da previsão. O resultado é um modelo que parece muito bom nas métricas de avaliação, mas falha completamente quando usado em produção. É um dos erros mais comuns e mais silenciosos em projetos de Machine Learning. A lista ``cols_nao_usar`` é uma forma explícita de documentar e proteger o modelo contra esse risco.

## 9. Split temporal (80% treino / 20% teste)

Para avaliar se um modelo realmente aprendeu padrões generalizáveis, e não apenas memorizou os dados de treino, é necessário separá-los em dois conjuntos: um para treinar o modelo e outro para testá-lo em dados que ele nunca viu.

In [ ]:
# Split temporal 80/20
train_size = int(len(df) * 0.8)

train_df = df.iloc[:train_size].copy()
test_df  = df.iloc[train_size:].copy()

#Separa entradas (X) e saída (y).
X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test  = test_df[feature_cols]
y_test  = test_df[target_col]

feature_cols, X_train.head()

(['dia',
  'mes',
  'ano',
  'dia_semana',
  'semana_ano',
  'mes_num',
  'fim_semana',
  'operadora_enc',
  'origem_enc',
  'destino_enc',
  'rota_enc'],
    dia  mes   ano  dia_semana  semana_ano  mes_num  fim_semana  operadora_enc  \
 0    1    1  2026           3           1        1           0              0   
 1    1    1  2026           3           1        1           0            113   
 2    1    1  2026           3           1        1           0            113   
 3    1    1  2026           3           1        1           0            113   
 4    1    1  2026           3           1        1           0            113   
 
    origem_enc  destino_enc  rota_enc  
 0         315          694      1633  
 1         608          778      2757  
 2         608          827      2762  
 3         608          834      2763  
 4         637          641      2934  )

**Por que divisão temporal e não aleatória?**

A divisão é feita de forma cronológica: os primeiros 80% dos registros (ordenados por data) formam o treino, e os últimos 20% formam o teste. Essa escolha é fundamental em dados temporais.

Se a divisão fosse aleatória, como faz o ``train_test_split`` padrão do sklearn, dias de dezembro poderiam cair no treino e dias de janeiro no teste, ou vice-versa. O modelo estaria "vendo o futuro" durante o treino, o que produziria métricas artificialmente boas que não se repetiriam na prática. Isso **é uma forma de data leakage temporal**.

A divisão cronológica simula exatamente o cenário real de uso: o modelo aprende com o passado e é avaliado na sua capacidade de prever o futuro, que é exatamente o que queremos.


**O que é ``.copy()``?**

O ``.copy()`` cria uma cópia independente de cada subconjunto. Sem ele, ``train_df`` e ``test_df`` seriam apenas "janelas" apontando para o DataFrame original, e qualquer modificação feita neles (como adicionar colunas) alteraria o df original também, causando bugs difíceis de rastrear.

**X e y: a convenção de Machine Learning**

A separação em X e y é uma convenção universal em Machine Learning:



*   X_train e X_test: as features (entradas do modelo); tudo que o modelo pode usar para fazer a previsão.

*   y_train e y_test: o target (saída esperada); o valor que o modelo deve aprender a prever.

Durante o treino, o modelo recebe X_train e y_train juntos e aprende a relação entre eles. Durante o teste, ele recebe apenas X_test e tenta prever y_test, sem vê-lo. Depois comparamos o que ele previu com o y_test real para medir o erro.

**80/20: por que essa proporção?**

Não existe uma regra universal, mas 80/20 é uma convenção amplamente usada que equilibra dois objetivos opostos: quanto mais dados no treino, melhor o modelo aprende; quanto mais dados no teste, mais confiável é a avaliação. Com datasets pequenos ou muito sazonais, às vezes faz sentido usar 70/30 ou até 60/40. Com datasets grandes, pode-se usar 90/10. Aqui, 80/20 é uma escolha razoável dado o volume de dados disponível.

## 10. Função de avaliação do modelo

Antes de treinar o modelo, definimos uma função reutilizável para calcular e exibir as métricas de avaliação. Ela é chamada tanto no treino quanto no teste, o que permite comparar os dois e identificar problemas como overfitting.

In [ ]:
def avaliar_modelo(model, X_train, y_train, X_test, y_test, nome="Modelo"):
    y_pred_train = model.predict(X_train)
    y_pred_test  = model.predict(X_test)

    mae_train  = mean_absolute_error(y_train, y_pred_train)
    mae_test   = mean_absolute_error(y_test, y_pred_test)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    rmse_test  = np.sqrt(mean_squared_error(y_test, y_pred_test))
    r2_train   = r2_score(y_train, y_pred_train)
    r2_test    = r2_score(y_test, y_pred_test)

    print(f"===== Desempenho do {nome} =====")
    print(f"MAE Treino: {mae_train:.2f}")
    print(f"MAE Teste : {mae_test:.2f}\n")
    print(f"RMSE Treino: {rmse_train:.2f}")
    print(f"RMSE Teste : {rmse_test:.2f}\n")
    print(f"R² Treino: {r2_train:.3f}")
    print(f"R² Teste : {r2_test:.3f}")

    return {
        "mae_train": mae_train, "mae_test": mae_test,
        "rmse_train": rmse_train, "rmse_test": rmse_test,
        "r2_train": r2_train, "r2_test": r2_test}

As três métricas usadas

*   MAE: Mean Absolute Error (Erro Médio Absoluto)

Calcula a média dos erros absolutos entre o valor previsto e o real:

``MAE = média(|real - previsto|)``

É a métrica mais intuitiva: se o MAE for 15, significa que, em média, o modelo erra 15 bilhetes para mais ou para menos. *Tem a vantagem de estar na mesma unidade que o target* (bilhetes), facilitando a interpretação. Trata todos os erros igualmente, independentemente do tamanho.

*   RMSE: Root Mean Squared Error (Raiz do Erro Quadrático Médio)

Calcula a raiz da média dos erros ao quadrado:

``RMSE = √(média((real - previsto)²))``

Ao elevar ao quadrado antes de tirar a média, o RMSE penaliza erros grandes de forma desproporcional. Um dia em que o modelo errou 100 bilhetes vai pesar muito mais do que dez dias em que errou 10 bilhetes cada. **Por isso o RMSE é sempre maior ou igual ao MAE**: quanto maior a diferença entre os dois, mais o modelo está errando em casos específicos. Também está na mesma unidade que o target.

*   R²: Coeficiente de Determinação

Mede quanto da variação do target o modelo consegue explicar, em uma escala de 0 a 1 (podendo ser negativo em casos muito ruins):

>>R² = 1.0: modelo perfeito, prevê exatamente o valor real em todos os casos.

>>R² = 0.8: o modelo explica 80% da variação, bom para dados reais de negócio.

>>R² = 0.0: o modelo não é melhor do que simplesmente prever a média do target para todos os dias.

>>R² negativo: o modelo é pior do que prever a média, algo está seriamente errado.

**Como interpretar treino vs teste juntos**

Avaliar o modelo nos dois conjuntos ao mesmo tempo é a forma de detectar **overfitting**, quando o modelo aprende os dados de treino tão bem que perde a capacidade de generalizar para dados novos:

| Situação | MAE Treino | MAE Teste | Diagnóstico |
| :--- | :--- | :--- | :--- |
| Ideal | Baixo | Baixo | Modelo generalizou bem |
| Overfitting | Muito baixo | Alto | Memorizou o treino, não aprendeu |
| Underfitting | Alto | Alto | Modelo muito simples, não aprendeu nada |

No Random Forest, um R² de treino próximo de 1.0 com R² de teste bem menor é o sinal clássico de overfitting: as árvores memorizaram os dados em vez de aprender os padrões.

## 11. Treinamento do modelo - Random Forest

Aqui decidi seguir com um modelo de Random Forest.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=2,
    random_state=0,
    n_jobs=-1
)

rf.fit(X_train, y_train)  # <-- PARTE 1: confirmar que o RF está treinado; aqui o modelo aprende; pipeline

RandomForestRegressor(max_depth=20, min_samples_leaf=2, n_estimators=300,
                      n_jobs=-1, random_state=0)

Para entender o Random Forest, é preciso entender primeiro a Decision Tree (Árvore de Decisão), que é o bloco fundamental do qual ele é construído.

Uma Decision Tree aprende fazendo perguntas binárias sobre os dados em sequência. Por exemplo:

>>"O dia é fim de semana?" >>> Sim >>> "A rota é CURITIBA - PONTA GROSSA?" >>> Sim >>> "Prevejo 45 bilhetes."

Cada pergunta divide os dados em dois grupos, e a árvore vai fazendo divisões até chegar a uma previsão. É intuitiva, rápida e fácil de interpretar: dá para visualizar cada decisão que o modelo tomou.

**O problema da Decision Tree: overfitting**

Se deixada crescer livremente, uma árvore de decisão vai criar divisões cada vez mais específicas **até memorizar** cada linha do dataset de treino, chegando a R² = 1.0 no treino e performance péssima no teste. Ela aprende até o ruído dos dados, não apenas os padrões reais. É como decorar os dados do 'futuro' antes mesmo de vê-lo.

É possível limitar o crescimento da árvore (via ``max_depth``, por exemplo), mas aí ela fica simples demais e perde capacidade preditiva. Existe um tradeoff difícil de equilibrar com uma única árvore.



**Funcionamento do Random Forest**

O Random Forest, por sua vez, é um modelo que resolve o problema do overfitting com duas ideias combinadas: bagging e aleatoriedade de features.


*   Bagging (Bootstrap Aggregating):

Em vez de treinar uma única árvore em todos os dados, o Random Forest **treina centenas de árvores**, aqui no caso, 300 (n_estimators=300). Cada árvore é treinada em uma amostra aleatória com reposição do dataset original (chamada de bootstrap sample). Isso significa que cada árvore vê um subconjunto *ligeiramente diferente* dos dados, aprende padrões *ligeiramente diferentes*, e comete erros diferentes.

*   Aleatoriedade de features

Além disso, em cada divisão de cada árvore, o algoritmo considera **apenas** um subconjunto aleatório das features disponíveis, não todas. Isso **força** as árvores a serem diferentes entre si, **evitando que todas aprendam a mesma coisa**.

**Previsão**

Para fazer uma previsão, todas as 300 árvores fazem sua previsão individualmente, e o resultado final é a média de todas elas. Erros individuais de cada árvore **tendem a se cancelar** quando você tira a média de muitas árvores independentes: esse **princípio** é chamado de **wisdom of crowds** (sabedoria das multidões). O resultado é um modelo muito mais estável e generalizável do que qualquer árvore individual.

Uma analogia útil: imagine que você quer estimar quantos bilhetes vai vender amanhã. Em vez de perguntar para uma única pessoa (Decision Tree), você pergunta para 300 pessoas com experiências e perspectivas ligeiramente diferentes (Random Forest) e tira a média das respostas. A média tende a ser mais confiável do que qualquer resposta individual.


Os parâmetros usados no código e o que cada um controla:

*   ``n_estimators=300``: número de árvores na floresta. Mais árvores = modelo mais estável e geralmente mais preciso, mas mais lento para treinar. 300 é um valor robusto para a maioria dos casos — acima de 500 o ganho costuma ser marginal.

*   ``max_depth=20``: profundidade máxima de cada árvore — quantas perguntas ela pode fazer em sequência antes de dar uma resposta. Limitar a profundidade é a principal forma de controlar o overfitting. Profundidade 20 é moderadamente alta; se o modelo overfitar, reduzir para 10 ou 15 é um bom próximo passo.

*   ``min_samples_leaf=2``: número mínimo de amostras que uma "folha" (nó terminal) da árvore precisa ter. Com valor 1 (padrão), cada folha pode representar um único dado de treino — overfitting garantido. Com valor 2, a árvore só faz uma divisão se os dois grupos resultantes tiverem pelo menos 2 exemplos cada, forçando um mínimo de generalização.

*   ``random_state=0``: semente aleatória que garante reprodutibilidade. Com o mesmo random_state, rodar o código duas vezes produz exatamente o mesmo modelo. Sem isso, cada execução geraria uma floresta diferente e os resultados seriam ligeiramente distintos a cada vez.

*   ``n_jobs=-1``: usa todos os núcleos de CPU disponíveis para treinar as árvores em paralelo. Como cada árvore é independente das outras, o treinamento é paralelizável — no Colab, isso pode reduzir o tempo de treino significativamente.

###Prós e contras do Random Forest

**Prós**:

*   Lida bem com features de tipos mistos (numéricas e categóricas codificadas);

*   Robusto a outliers e dados faltantes (com tratamento mínimo);

*   Não exige normalização das features (ao contrário de modelos como Ridge Regression ou Redes Neurais);

*   Fornece importância das features nativamente, permitindo entender quais variáveis mais influenciam a previsão;

*   Funciona bem mesmo sem muito ajuste de hiperparâmetros;

*   Lida bem com relações não-lineares entre features e target.

**Contras**:

*   Não extrapola: o Random Forest só consegue prever valores dentro do intervalo **que já viu no treino**. Se o maior valor de bilhetes no treino foi 80, ele nunca vai prever 90, mesmo que o padrão dos dados sugira crescimento. Isso é uma limitação importante para previsões de longo prazo ou em contextos de crescimento acelerado.

*   Pouca interpretabilidade: ao contrário de uma única Decision Tree, **não é possível** visualizar o raciocínio de 300 árvores **ao mesmo tempo**. É um modelo do tipo "caixa preta": sabemos o que entra e o que sai, mas o caminho interno é opaco.

*   Pesado em memória: 300 árvores com profundidade 20 podem ocupar bastante memória RAM, especialmente com datasets grandes.

*   Lento para prever em escala: fazer previsões para milhões de linhas pode ser lento, pois cada linha precisa percorrer todas as 300 árvores.

| Situação | Alternativa recomendada | Por quê |
| :--- | :--- | :--- |
| Dataset muito grande (milhões de linhas) | LightGBM ou XGBoost | Muito mais rápidos e eficientes em memória, com performance similar ou superior |
| Previsão de série temporal com tendência ou sazonalidade forte | Prophet (Meta) ou ARIMA | Modelam explicitamente tendência e sazonalidade ao longo do tempo |
| Você precisa explicar cada previsão individualmente | Decision Tree simples ou Regressão Linear | Muito mais interpretáveis |
| O target cresce ao longo do tempo (tendência) | LightGBM com features de lag ou Prophet | Random Forest não extrapola além do máximo visto no treino |
| Poucos dados (menos de algumas centenas de linhas) | Regressão Linear com regularização (Ridge/Lasso) | Random Forest precisa de volume para funcionar bem |


No nosso caso, o Random Forest é uma escolha adequada porque:



*   os dados não têm uma tendência de crescimento forte que exija extrapolação,

*   as features são mistas (categóricas + temporais), e;

*   o volume de dados é suficiente para que o modelo aprenda padrões por rota e por período.

## 12. Avaliação final

Com o modelo treinado, chamamos a função avaliar_modelo definida anteriormente para calcular as métricas nos dois conjuntos (treino e teste) e diagnosticar a qualidade do aprendizado.

In [ ]:
resultados_rf = avaliar_modelo(rf, X_train, y_train, X_test, y_test, nome="Random Forest")

===== Desempenho do Random Forest =====
MAE Treino: 0.48
MAE Teste : 0.79

RMSE Treino: 0.91
RMSE Teste : 1.44

R² Treino: 0.755
R² Teste : 0.466


Os resultados aqui são o primeiro indicador real de se o pipeline está funcionando corretamente antes de partir para as previsões futuras. Alguns pontos a observar:


*   Se o R² de treino for muito próximo de 1.0 e o R² de teste for significativamente menor, **há overfitting**: o modelo memorizou o treino. Nesse caso, reduzir ``max_depth`` ou aumentar ``min_samples_leaf`` são os primeiros ajustes a tentar.

*   Se ambos os R² forem baixos, há **underfitting**: o modelo não aprendeu padrões suficientes. Adicionar mais features ou aumentar ``n_estimators`` pode ajudar.

*   Se o MAE de teste for compatível com a escala do negócio (por exemplo, errar em média 10 bilhetes num contexto onde dias típicos têm 200 vendas é razoável) o modelo **está em um nível utilizável** para orientar decisões.

# Definição do horizonte de previsão

Antes de prever qualquer coisa, é necessário definir para quais datas queremos as previsões. Essa etapa cria o eixo temporal futuro, ou seja, as datas que ainda não existem no dataset histórico.

In [ ]:
# Quantos dias você quer prever?
horizonte = 90  # pode trocar para 30, 60, 180...

ultima_data = df["data"].max() # última data disponível no banco; Busca a data mais recente do dataset histórico

# Criação das datas futuras
datas_futuras = pd.date_range(
    ultima_data + pd.Timedelta(days=1),
    periods=horizonte,
    freq="D"
)

datas_futuras[:5]

DatetimeIndex(['2026-05-18', '2026-05-19', '2026-05-20', '2026-05-21',
               '2026-05-22'],
              dtype='datetime64[ns]', freq='D')

**O que cada linha faz**

*   ``horizonte = 90``: define quantos dias à frente queremos prever. Aqui, 90 dias (3 meses). Vale lembrar que quanto maior o horizonte, menos confiável tende a ser a previsão: o modelo extrapola padrões do passado, e quanto mais longe no futuro, mais esses padrões podem não se repetir da mesma forma.

*   ``ultima_data = df["data"].max()``: busca a data mais recente do histórico. A previsão começa no dia seguinte a essa data, garantindo que não haja sobreposição entre histórico e futuro.

*   ``pd.date_range(...)``: gera uma sequência de datas diárias (freq="D") começando em ultima_data + 1 dia e com horizonte períodos. O resultado é um array com exatamente 90 datas consecutivas, uma por dia.


**O que essa etapa não faz**

É importante entender que essa etapa apenas cria o eixo do tempo, ou seja, uma lista de datas. Ela não prevê nada ainda. Para prever, o modelo precisa de features completas para cada data futura (dia da semana, rota, operadora etc.), que serão construídas na próxima etapa. É a mesma lógica de preparar uma tabela vazia com as datas nas linhas antes de preenchê-la com os dados que o modelo vai usar como entrada.


**Por que não simplesmente passar as datas diretamente para o modelo?**

Como vimos na Seção 8, a coluna data foi excluída das features, com o modelo não recebendo datas brutas. Ele recebe *features derivadas* da data: dia_semana, semana_ano, mes_num e fim_semana. Portanto, as datas futuras aqui servem como ponto de partida para calcular essas features na próxima etapa, **não como input** direto do modelo.

=======================================================================================================================================================================================================

# Elementos para previsão: Construção do dataset futuro (df_future)

Essa é uma das etapas mais importantes (e mais delicadas) de todo o pipeline. O objetivo é construir um DataFrame com todas as combinações de datas futuras e rotas reais, no mesmo formato que o modelo viu durante o treino.

In [ ]:
# FUTURO (TOTAL GERAL), somente rotas reais e operadoras reais
# Substitui:
# 1) bloco de top_rotas
# 2) bloco de df_future usando operadora_padrao

# 1) Pegar TODAS as combinações reais (rota × operadora) que existiram no histórico
# (não cria produto cartesiano de origem × destino)
combos_reais = (
    df[["rota", "origem", "destino", "operadora"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# (opcional) sanity check: quantas combinações existem
print("Combinações reais (rota × operadora):", len(combos_reais))

# 2) Construir df_future cruzando datas futuras com as combinações reais
future_rows = []
for data_future in datas_futuras:
    for _, row in combos_reais.iterrows():
        future_rows.append({
            "data": data_future,
            "dia": data_future.day,
            "mês": data_future.month,
            "ano": data_future.year,
            "origem": row["origem"],
            "destino": row["destino"],
            "rota": row["rota"],           # já vem pronta do histórico
            "operadora": row["operadora"]  # operadora real daquela rota
        })

df_future = pd.DataFrame(future_rows)

# (opcional) pra conferir rapidamente
df_future.head()

Combinações reais (rota × operadora): 4715


,data,dia,mês,ano,origem,destino,rota,operadora
0,2026-05-18,18,5,2026,IGUABA GRANDE - RJ,RIO DE JANEIRO - NOVO RIO - RJ,IGUABA GRANDE - RJ - RIO DE JANEIRO - NOVO RIO...,1001
1,2026-05-18,18,5,2026,PONTA GROSSA - PR,SAO MIGUEL DO OESTE - SC,PONTA GROSSA - PR - SAO MIGUEL DO OESTE - SC,PRINCESA DOS CAMPOS
2,2026-05-18,18,5,2026,PONTA GROSSA - PR,TELEMACO BORBA - PR,PONTA GROSSA - PR - TELEMACO BORBA - PR,PRINCESA DOS CAMPOS
3,2026-05-18,18,5,2026,PONTA GROSSA - PR,TIBAGI - PR,PONTA GROSSA - PR - TIBAGI - PR,PRINCESA DOS CAMPOS
4,2026-05-18,18,5,2026,PRUDENTOPOLIS - PR,PONTA GROSSA - PR,PRUDENTOPOLIS - PR - PONTA GROSSA - PR,PRINCESA DOS CAMPOS


A variável ``combos_reais`` é fundamental. Uma alternativa ingênua seria fazer o **produto cartesiano** de todas as origens com todos os destinos, ou seja, assumir que qualquer cidade pode se conectar a qualquer outra. **Mas isso criaria rotas que nunca existiram (ex: "CURITIBA - MANAUS" se nunca houve venda dessa rota), e o modelo não teria como prever corretamente para combinações que nunca viu no treino**.

>>Ao usar apenas ``combos_reais``, garantimos que o modelo só vai prever para rotas e operadoras que ele conhece, preservando a estrutura real do negócio.

**O loop de construção**

O loop duplo (``for data_future`` × ``for _, row in combos_reais``) cria uma linha para cada combinação de data futura com cada combo real de rota × operadora. Se houver 1.084 combinações reais e 90 dias futuros, o resultado é um DataFrame com 97.560 linhas, sendo uma para cada (data, rota, operadora) possível.

Cada linha já inclui as informações básicas necessárias para calcular as features do modelo na etapa seguinte: data, dia, mês, ano, origem, destino, rota e operadora.

**O problema que isso introduz e como será resolvido**

Ao assumir que todas as 1.084 combinações existem em todos os 90 dias futuros, estamos criando um cenário otimista demais: na realidade, um dia típico do histórico tinha em média apenas 121 combinações ativas, não 1.084. A maioria das combinações fica inativa na maior parte dos dias.

Isso explica o problema de inflação visto anteriormente: o modelo prevê um número positivo de bilhetes para cada uma das 1.084 linhas de um dia, e quando somamos tudo, o total diário fica muito acima da realidade, como os 1.700 bilhetes previstos contra 250 bilhetes reais observados no histórico.

A causa raiz é que o modelo foi treinado apenas em linhas onde algo aconteceu, nunca viu um registro com bilhetes = 0. Então ele não aprendeu a prever zero para combinações inativas, e tende a prever sempre algum valor positivo.

A solução correta, que será implementada na seção "Versão com todas as rotas", é transformar o treino em um painel completo, preenchendo com bilhetes = 0 os dias em que cada combinação não teve venda. Assim o modelo aprende tanto o "quando vende" quanto o "quando não vende", e as previsões passam a respeitar a proporção real de cada rota no mix total de vendas.

# Pré-processamento do dataset futuro

Com o ``df_future`` construído, é necessário aplicar exatamente as mesmas transformações que foram aplicadas no histórico antes do treino.

>>**Esse princípio (de aplicar no futuro tudo que foi aplicado no passado, na mesma ordem) é uma das regras mais importantes de qualquer pipeline de Machine Learning**.

In [ ]:
# Mesma limpeza de texto
for col in ["operadora", "origem", "destino"]:
    df_future[col] = df_future[col].apply(normalizar_texto)

# Rota
df_future["rota"] = df_future["origem"] + " - " + df_future["destino"] # Reconstrói a coluna rota no mesmo formato do histórico (ex: "SAO PAULO - CAMPINAS"); Necessário para depois gerar rota_enc com le_rota.transform(...).

# Recalcular features de tempo (garantia); Recria as features de tempo para o futuro (igual no treino). A frase “garantia” significa: mesmo que já existam colunas, recalcular evita inconsistência.
df_future["dia_semana"] = df_future["data"].dt.weekday
df_future["semana_ano"] = df_future["data"].dt.isocalendar().week.astype(int)
df_future["mes_num"]    = df_future["mês"]
df_future["fim_semana"] = df_future["dia_semana"].isin([5, 6]).astype(int)

df_future.head()

,data,dia,mês,ano,origem,destino,rota,operadora,dia_semana,semana_ano,mes_num,fim_semana
0,2026-05-18,18,5,2026,IGUABA GRANDE - RJ,RIO DE JANEIRO - NOVO RIO - RJ,IGUABA GRANDE - RJ - RIO DE JANEIRO - NOVO RIO...,1001,0,21,5,0
1,2026-05-18,18,5,2026,PONTA GROSSA - PR,SAO MIGUEL DO OESTE - SC,PONTA GROSSA - PR - SAO MIGUEL DO OESTE - SC,PRINCESA DOS CAMPOS,0,21,5,0
2,2026-05-18,18,5,2026,PONTA GROSSA - PR,TELEMACO BORBA - PR,PONTA GROSSA - PR - TELEMACO BORBA - PR,PRINCESA DOS CAMPOS,0,21,5,0
3,2026-05-18,18,5,2026,PONTA GROSSA - PR,TIBAGI - PR,PONTA GROSSA - PR - TIBAGI - PR,PRINCESA DOS CAMPOS,0,21,5,0
4,2026-05-18,18,5,2026,PRUDENTOPOLIS - PR,PONTA GROSSA - PR,PRUDENTOPOLIS - PR - PONTA GROSSA - PR,PRINCESA DOS CAMPOS,0,21,5,0


**Por que isso é tão crítico?**


O modelo aprendeu padrões a partir de dados que passaram por
*   normalização de texto,
*   criação de features de tempo e
*   encoding.

Se os dados futuros chegarem ao modelo em um formato diferente (mesmo que sutilmente diferente) o modelo vai "ver" categorias que não reconhece ou features com significado diferente do que aprendeu, e as previsões serão incorretas ou o código vai quebrar. É a mesma lógica de um tradutor que aprendeu a traduzir texto em maiúsculas sem acento: diante de um texto em minúsculas com acento, ele não vai reconhecer as palavras.

**O que cada bloco faz**


*   Normalização de texto:

Aplica a mesma função ``normalizar_texto`` nas colunas operadora, origem e destino do dataset futuro. Como essas strings vieram do próprio histórico (via ``combos_reais``), na prática elas já estão normalizadas, mas reaplicar é uma garantia defensiva. Se em algum momento o código de construção do ``df_future`` for alterado e as strings vierem de outra fonte, a normalização já estará no lugar.

*   Reconstrução da coluna rota:

A coluna rota é reconstruída concatenando origem e destino, mesmo que já venha pronta do histórico. Isso garante que o formato seja idêntico ao que o ``le_rota`` aprendeu durante o treino ("ORIGEM - DESTINO"). Uma diferença de um espaço ou maiúscula já causaria erro no encoding.

*   Features de tempo:

``dia_semana``, ``semana_ano``, ``mes_num`` e ``fim_semana`` são recalculadas a partir das datas futuras. Essas features não existiam no ``df_future`` ainda, precisam ser derivadas da coluna data exatamente como foram no histórico. São elas que vão permitir ao modelo identificar, por exemplo, que uma sexta-feira de dezembro futura tem um perfil similar às sextas-feiras de dezembro do passado.

**Observação**

Vou renomear a coluna de mês para mes porque estou usando uma nova base em que esse cabeçalho muda de nome.

In [ ]:
df = df.rename(columns={'mês': 'mes'})

In [ ]:
df.head(3)

,dia,mes,ano,operadora,origem,destino,canal_de_venda,bilhetes,ordens,receita,...,data,rota,dia_semana,semana_ano,mes_num,fim_semana,operadora_enc,origem_enc,destino_enc,rota_enc
0,1,1,2026,1001,IGUABA GRANDE - RJ,RIO DE JANEIRO - NOVO RIO - RJ,App,4,1,35.3872,...,2026-01-01,IGUABA GRANDE - RJ - RIO DE JANEIRO - NOVO RIO...,3,1,1,0,0,315,694,1633
1,1,1,2026,PRINCESA DOS CAMPOS,PONTA GROSSA - PR,SAO MIGUEL DO OESTE - SC,App,1,1,21.1600,...,2026-01-01,PONTA GROSSA - PR - SAO MIGUEL DO OESTE - SC,3,1,1,0,113,608,778,2757
2,1,1,2026,PRINCESA DOS CAMPOS,PONTA GROSSA - PR,TELEMACO BORBA - PR,Web,1,1,18.3334,...,2026-01-01,PONTA GROSSA - PR - TELEMACO BORBA - PR,3,1,1,0,113,608,827,2762


# “Máscara de válidos” (filtro contra categorias desconhecidas)

Antes de aplicar o encoding no dataset futuro, precisamos garantir que todas as categorias que aparecem nele existiam também no histórico de treino. Essa etapa faz exatamente essa verificação.

In [ ]:
mascara_validos = (
    df_future["origem"].isin(le_origem.classes_) &
    df_future["destino"].isin(le_destino.classes_) &
    df_future["rota"].isin(le_rota.classes_) &
    df_future["operadora"].isin(le_operadora.classes_)
)

df_future_valid = df_future[mascara_validos].copy()
print("Linhas futuras válidas:", len(df_future_valid), "de", len(df_future))

Linhas futuras válidas: 424350 de 424350


**O problema das categorias desconhecidas**

O ``LabelEncoder`` aprende um mapeamento fixo durante o ``.fit_transform()`` no histórico. Se uma categoria nova aparecer no ``.transform()`` (uma origem, destino, rota ou operadora que não existia no treino) ele lança um erro e interrompe a execução.

No nosso caso, como o ``df_future`` foi construído a partir do próprio histórico (``combos_reais``), todas as categorias já deveriam ser conhecidas. Mas essa verificação existe como proteção: se o pipeline for reaproveitado com dados diferentes, ou se houver alguma inconsistência na normalização de texto, a máscara captura o problema antes que ele quebre o código silenciosamente.

**Como a máscara funciona**

``le_origem.classes_`` é um array com todos os valores únicos que o encoder aprendeu no treino. O ``.isin()`` retorna uma série booleana (True/False) para cada linha do ``df_future``, indicando se aquela categoria existe no encoder.
O operador ``&`` combina as quatro verificações (origem, destino, rota e operadora) exigindo que todas sejam válidas simultaneamente para que a linha seja mantida. Uma linha com origem válida mas operadora desconhecida seria descartada. O resultado é ``df_future_valid``, contendo apenas as linhas que o encoder consegue processar com segurança.

**O que o print revela**

A comparação "Linhas futuras válidas: X de Y" é um diagnóstico útil:

*   Se X = Y: todas as combinações do futuro existiam no histórico. Pipeline consistente.

*   Se X < Y: algumas combinações foram descartadas. Vale investigar quais categorias não foram reconhecidas; pode indicar problema de normalização ou uma rota que apareceu no futuro mas não tinha histórico suficiente.

# Encoding do futuro e geração de previsões

### Encoding

Com as categorias validadas, aplicamos o encoding nas colunas categóricas do ``df_future_valid`` usando os mesmos encoders treinados no histórico, so que, desta vez, com ``.transform()`` em vez de ``.fit_transform()``.

In [ ]:
df_future_valid["operadora_enc"] = le_operadora.transform(df_future_valid["operadora"])
df_future_valid["origem_enc"]    = le_origem.transform(df_future_valid["origem"])
df_future_valid["destino_enc"]   = le_destino.transform(df_future_valid["destino"])
df_future_valid["rota_enc"]      = le_rota.transform(df_future_valid["rota"])

Esse ponto merece atenção: estamos usando os mesmos objetos ``le_operadora``, ``le_origem``, ``le_destino`` e ``le_rota`` que foram criados na Seção 7. Eles já sabem o mapeamento de categoria para número que aprenderam no histórico, e aqui apenas aplicam esse mapeamento nos dados futuros. Isso **garante** que "CURITIBA" no futuro receba exatamente o mesmo número que "CURITIBA" recebeu no treino, pois, sem isso, o modelo estaria recebendo números com significados diferentes dos que aprendeu.

### Montando X_future e gerando previsões

In [ ]:
X_future = df_future_valid[feature_cols]

df_future_valid["bilhetes_previstos"] = rf.predict(X_future)

# (opcional) bilhetes não pode ser negativo; e faz sentido arredondar
df_future_valid["bilhetes_previstos"] = (
    df_future_valid["bilhetes_previstos"]
    .clip(lower=0)
    .round()
    .astype(int)
)

KeyError: "['mes'] not in index"

``X_future`` é montado selecionando exatamente as mesmas colunas de ``feature_cols`` definidas na Seção 8, garantindo que o modelo receba as features na mesma ordem e formato que viu no treino.

O ``rf.predict(X_future)`` percorre as 300 árvores para cada linha do ``df_future_valid`` e retorna a média das previsões individuais; um número de bilhetes estimado para cada combinação de (data, rota, operadora).

**Pós-processamento das previsões**

Dois ajustes são aplicados nas previsões brutas:

*   ``.clip(lower=0)``: **garante que nenhuma previsão seja negativa**. O Random Forest pode ocasionalmente prever valores negativos para combinações raras ou atípicas, o que não faz sentido para bilhetes. O clip força o mínimo para zero.

*   ``.round().astype(int)``: arredonda para o inteiro mais próximo e converte para número inteiro. Bilhetes são unidades discretas

**O problema que persiste**

Mesmo com o encoding correto e o pós-processamento, o problema de inflação das previsões discutido na Seção de 'Construção de dataset futuro' ainda existe aqui. O modelo está prevendo um valor positivo para praticamente todas as 1.084 combinações em todos os dias, quando na realidade a maioria deveria prever zero. A soma diária por isso continua inflada em relação ao histórico real. **Esse é exatamente o problema que a seção "Versão com todas as rotas" vai atacar**.

# Agregação por dia e visualização

Até aqui, o modelo gerou uma previsão de bilhetes para cada linha do ``df_future_valid``, ou seja, para cada combinação de (data, rota, operadora). Para obter o total diário previsto, somamos todas essas previsões agrupando por data:

In [ ]:
previsao_por_dia = (
    df_future_valid
    .groupby("data")["bilhetes_previstos"]
    .sum()
    .reset_index()
)
previsao_por_dia.head()

KeyError: 'Column not found: bilhetes_previstos'

Da mesma forma, agregamos o histórico real por dia para ter uma base de comparação:

In [ ]:
historico_por_dia = (
    df.groupby("data")["bilhetes"]
      .sum()
      .reset_index()
      .rename(columns={"bilhetes": "bilhetes_reais"}))

In [ ]:
print("Histórico (últimos 7 dias):")
print(historico_por_dia.tail(7))

print("\nPrevisão (primeiros 7 dias):")
print(previsao_por_dia.head(7))

Histórico (últimos 7 dias):
         data  bilhetes_reais
18 2025-11-19             273
19 2025-11-20             217
20 2025-11-21             253
21 2025-11-22             231
22 2025-11-23             261
23 2025-11-24             198
24 2025-11-25             183

Previsão (primeiros 7 dias):
        data  bilhetes_previstos
0 2025-11-26                1719
1 2025-11-27                1690
2 2025-11-28                1711
3 2025-11-29                1743
4 2025-11-30                1782
5 2025-12-01                1717
6 2025-12-02                1625


Essa comparação entre o último período do histórico e o primeiro período da previsão **é o primeiro teste de sanidade do modelo**: os valores previstos deveriam estar em uma faixa compatível com os valores históricos recentes. Uma diferença muito grande, como a que foi observada aqui, com histórico em torno de 250 bilhetes/dia e previsão em torno de 1.700 bilhetes/dia.

>>Isso é um sinal claro de que **algo está errado no pipeline**, independentemente das métricas de avaliação do modelo.

**Por que as métricas do modelo podem estar boas mas a previsão estar errada?**

As métricas calculadas para R^2, MAE, RMSE avaliam o modelo por linha, ou seja, quão bem ele prevê o número de bilhetes de uma combinação específica de (data, rota, operadora) em um único dia. Se o modelo aprendeu bem esse comportamento individual, as métricas ficam boas.

Mas **o problema surge na agregação**: quando somamos as previsões de todas as 1.084 combinações para um único dia, pequenos erros positivos em cada linha se acumulam em um erro enorme no total. Se o modelo tende a superestimar mesmo que levemente (o que acontece quando nunca viu zeros no treino) esse viés se multiplica por 1.084 linhas e produz totais completamente fora da realidade.
É a diferença entre um modelo que prevê bem individualmente, mas mal no agregado, e o que importa para a operação do negócio é o agregado.

#Diagnóstico: o problema do painel incompleto

In [ ]:
# quantas combinações (rota, operadora) existem por dia no histórico?
combos_por_dia = df.groupby("data").apply(lambda x: x[["rota","operadora"]].drop_duplicates().shape[0])
print(combos_por_dia.describe())

print("Total combos únicos no histórico:", df[["rota","operadora"]].drop_duplicates().shape[0])


count     25.000000
mean     121.120000
std       20.618196
min       89.000000
25%      106.000000
50%      117.000000
75%      133.000000
max      168.000000
dtype: float64
Total combos únicos no histórico: 1084


/tmp/ipykernel_7122/296493760.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  combos_por_dia = df.groupby("data").apply(lambda x: x[["rota","operadora"]].drop_duplicates().shape[0])


Essa etapa revela numericamente por que a previsão inflou tanto.

>>O código calcula **quantas combinações únicas de (rota × operadora) existiram em cada dia do histórico**, e compara com o total de combinações únicas em todo o período.

Os resultados indicam que:

*   O histórico tem 1.084 combinações únicas de rota × operadora ao longo de todo o período.

*   Em um dia típico, apenas cerca de 121 dessas combinações estavam ativas, cerca de 11% do total.

*   No melhor dia, havia 168 combinações ativas. No pior, apenas 89.

Ou seja: **o histórico não é um painel completo**. Um painel completo seria aquele em que todas as 1.084 combinações aparecem em todos os dias, com bilhetes = 0 nos dias sem venda. O que temos é um painel esparso: só aparecem no dataset os dias em que houve pelo menos uma venda para aquela combinação.

**Por que o painel esparso é um problema para o modelo**

Quando o modelo foi treinado nesse painel esparso, ele nunca viu um zero. Todas as linhas do treino tinham pelo menos 1 bilhete vendido, porque só existem no dataset linhas em que algo aconteceu.

>>O modelo aprendeu a prever "quanto se vende quando se vende", mas não aprendeu "quando não se vende nada".

Quando construímos o ``df_future`` com todas as 1.084 combinações para todos os 90 dias, estamos pedindo ao modelo que preveja para combinações que, na maioria dos dias, deveriam receber bilhetes = 0. Como ele nunca aprendeu esse comportamento, prevê algum valor positivo para quase todas as linhas, e a soma explode.


**A solução: painel completo com zeros no treino**

A correção é transformar o treino em um painel completo: para cada dia do histórico, todas as 1.084 combinações devem estar presentes, com bilhetes = 0 nos dias sem venda. Assim o modelo aprende dois comportamentos distintos:

*   Quando prever zero: a maioria dos dias, para a maioria das combinações.

*   Quando prever valores positivos: nos dias em que aquela rota específica tende a vender, na proporção real do histórico.

Esse é exatamente o objetivo da próxima seção ("Versão com todas as rotas") onde o pipeline será corrigido do zero com essa abordagem.

Vale notar também que esse painel completo com zeros resolve **naturalmente** o problema da seguinte ponderação:
>>**rotas com 60 bilhetes/dia vão ter muito mais linhas com valores altos do que rotas com 2 bilhetes/dia, que terão a maioria das linhas com zero**.

O modelo aprende automaticamente essa proporção sem nenhuma reponderação artificial, é o próprio volume de dados que carrega o peso correto de cada rota.

============================================================================================================================================================================================================================================================================================================================================================================================================================================

#COMPLETO - Versão com todas as rotas: construção do painel completo

Antes de seguirmos, precisamos verificar se, no dataset, o período histórico é contínuo (todos os dias do calendário têm pelo menos alguma venda) ou existem dias sem nenhuma venda em nenhuma rota.

*   Se o histórico for contínuo (nenhum dia sem venda): podemos construir o painel completo com zeros de forma direta, cruzando todas as datas do histórico com todas as 1.084 combinações.

*   Se houver dias sem venda: precisamos decidir se esses dias entram no painel com zeros em todas as combinações (o que faz sentido se foram dias sem operação (feriados, paralisações)) ou se são lacunas de dados que não deveriam ser tratadas como zero.

In [ ]:
# Verificar se há dias sem nenhuma venda no histórico
todas_as_datas = pd.date_range(df["data"].min(), df["data"].max(), freq="D")
datas_no_historico = df["data"].unique()

datas_faltantes = todas_as_datas[~todas_as_datas.isin(datas_no_historico)]

print(f"Período do histórico: {df['data'].min().date()} até {df['data'].max().date()}")
print(f"Total de dias no calendário: {len(todas_as_datas)}")
print(f"Total de dias com pelo menos 1 venda: {len(datas_no_historico)}")
print(f"Dias sem nenhuma venda: {len(datas_faltantes)}")

if len(datas_faltantes) > 0:
    print("\nDatas sem venda:")
    print(datas_faltantes)
else:
    print("\nNenhum dia sem venda; histórico contínuo.")

Período do histórico: 2025-11-01 até 2025-11-25
Total de dias no calendário: 25
Total de dias com pelo menos 1 venda: 25
Dias sem nenhuma venda: 0

Nenhum dia sem venda; histórico contínuo.


Temos histórico contínuo, ou seja, 25 dias, todos com pelo menos 1 venda.

As seções anteriores revelaram o problema central: **o modelo foi treinado apenas em dias com venda, nunca vendo zeros**. Ao prever para todas as 1.084 combinações em todos os dias futuros, o total diário inflou de 250 para 1.700 bilhetes; completamente fora da realidade.

A correção consiste em transformar o treino em um painel completo: para cada dia do histórico, todas as combinações reais de rota × operadora estão presentes: com bilhetes = 0 nos dias sem venda para aquela combinação. Assim **o modelo aprende tanto "quando vende" quanto "quando não vende nada"**, e as previsões futuras passam a respeitar a proporção real de cada rota no mix total de vendas.

Rotas com alto volume histórico vão ter muitas linhas com valores positivos altos. Rotas raras vão ter a maioria das linhas com zero e poucas linhas com valores pequenos. O modelo aprende essa proporção automaticamente: **é o próprio volume de dados que carrega o peso correto de cada rota, sem nenhuma reponderação artificial**.

##Construção do painel completo

O painel completo é construído cruzando todas as datas do histórico com todas as combinações reais de rota × operadora. O resultado é um DataFrame onde cada linha representa um (dia, rota, operadora), existindo ou não venda naquele dia para aquela combinação.

In [ ]:
# 1) Todas as combinações reais de rota × operadora do histórico
combos_reais = (
    df[["rota", "origem", "destino", "operadora"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# 2) Todas as datas do histórico
datas_historico = pd.date_range(df["data"].min(), df["data"].max(), freq="D")

# 3) Produto completo: cada data × cada combo real
painel_rows = []
for data in datas_historico:
    for _, row in combos_reais.iterrows():
        painel_rows.append({
            "data"      : data,
            "origem"    : row["origem"],
            "destino"   : row["destino"],
            "rota"      : row["rota"],
            "operadora" : row["operadora"]
        })

df_painel = pd.DataFrame(painel_rows)

print(f"Linhas no painel completo: {len(df_painel)}")
print(f"Esperado: {len(datas_historico)} dias × {len(combos_reais)} combos = {len(datas_historico) * len(combos_reais)}")

Linhas no painel completo: 27100
Esperado: 25 dias × 1084 combos = 27100


Com 25 dias e 1.084 combinações, o painel terá 27.100 linhas, contra as linhas esparsas do histórico original, que só existiam quando havia venda. Essa é a diferença fundamental: agora o modelo vai ver explicitamente os zeros.
Note que usamos combos_reais (apenas combinações que existiram ao menos uma vez no histórico), e não o produto cartesiano de todas as origens com todos os destinos. **Isso garante que não criamos rotas fictícias que nunca operaram**.

##Preenchendo os zeros



In [ ]:
# 4) Fazer o merge com o histórico real para trazer os bilhetes
#    Combinações sem venda naquele dia ficam com NaN > preenchemos com 0
df_painel = df_painel.merge(
    df[["data", "rota", "operadora", "bilhetes"]],
    on=["data", "rota", "operadora"],
    how="left"
)

df_painel["bilhetes"] = df_painel["bilhetes"].fillna(0).astype(int)

print(f"Zeros inseridos: {(df_painel['bilhetes'] == 0).sum()}")
print(f"Linhas com venda: {(df_painel['bilhetes'] > 0).sum()}")
print(f"\nDistribuição de bilhetes:")
print(df_painel["bilhetes"].describe())

Zeros inseridos: 24072
Linhas com venda: 3037

Distribuição de bilhetes:
count    27109.000000
mean         0.235309
std          1.221759
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         34.000000
Name: bilhetes, dtype: float64


Com o painel completo construído, fazemos um left join com o histórico real trazendo os bilhetes reais para cada (data, rota, operadora). Combinações que não tiveram venda naquele dia ficam com NaN, que preenchemos com zero via .``fillna(0)``.

O ``describe()`` ao final é um diagnóstico importante: a média de bilhetes no painel completo vai ser muito menor do que a média do histórico esparso original, porque agora os zeros estão sendo considerados. Isso é esperado e correto, pois reflete a realidade de que a maioria das combinações fica inativa na maioria dos dias.

Se a proporção de zeros for muito alta (acima de 95%, por exemplo), vale considerar se faz sentido incluir todas as 1.084 combinações ou se algumas são tão raras que seria melhor filtrá-las por um mínimo de dias com venda no histórico.

##Features de tempo e encoding

In [ ]:
# 5) Recria as features de tempo no painel
df_painel["dia"]        = df_painel["data"].dt.day
df_painel["mês"]        = df_painel["data"].dt.month
df_painel["ano"]        = df_painel["data"].dt.year
df_painel["dia_semana"] = df_painel["data"].dt.weekday
df_painel["semana_ano"] = df_painel["data"].dt.isocalendar().week.astype(int)
df_painel["mes_num"]    = df_painel["mês"]
df_painel["fim_semana"] = df_painel["dia_semana"].isin([5, 6]).astype(int)

# 6) Encoding das categorias
le_op2   = LabelEncoder()
le_ori2  = LabelEncoder()
le_dest2 = LabelEncoder()
le_rot2  = LabelEncoder()

df_painel["operadora_enc"] = le_op2.fit_transform(df_painel["operadora"])
df_painel["origem_enc"]    = le_ori2.fit_transform(df_painel["origem"])
df_painel["destino_enc"]   = le_dest2.fit_transform(df_painel["destino"])
df_painel["rota_enc"]      = le_rot2.fit_transform(df_painel["rota"])

df_painel.head()

,data,origem,destino,rota,operadora,bilhetes,dia,mês,ano,dia_semana,semana_ano,mes_num,fim_semana,operadora_enc,origem_enc,destino_enc,rota_enc
0,2025-11-01,BALNEARIO CAMBORIU,CURITIBA,BALNEARIO CAMBORIU - CURITIBA,BRASIL SUL,2,1,11,2025,5,44,11,1,5,21,80,32
1,2025-11-01,FLORIANOPOLIS,IMBITUBA,FLORIANOPOLIS - IMBITUBA,SANTO ANJO,1,1,11,2025,5,44,11,1,64,77,113,316
2,2025-11-01,CRICIUMA,TUBARAO,CRICIUMA - TUBARAO,SANTO ANJO,2,1,11,2025,5,44,11,1,64,67,281,188
3,2025-11-01,TRES PINHEIROS,GUARAPUAVA,TRES PINHEIROS - GUARAPUAVA,PRINCESA DOS CAMPOS,1,1,11,2025,5,44,11,1,51,245,106,936
4,2025-11-01,TOLEDO,VILA NOVA,TOLEDO - VILA NOVA,PRINCESA DOS CAMPOS,1,1,11,2025,5,44,11,1,51,241,295,930


As features de tempo e o encoding são recriados no painel completo exatamente como foram feitos no histórico original; mesma lógica, mesma ordem. A única diferença é que usamos novos objetos de LabelEncoder (le_op2, le_ori2 etc.) para não sobrescrever os encoders originais, que ainda podem ser úteis para comparação.

É importante criar encoders novos aqui porque o painel completo pode ter uma distribuição de categorias ligeiramente diferente do histórico esparso original, e queremos que o encoding reflita exatamente o que o novo modelo vai ver no treino.

#Split temporal do painel completo

O split segue a mesma lógica cronológica da Seção em que fizemos isso pela primeira vez: os primeiros 80% das linhas formam o treino e os últimos 20% formam o teste. Mas há uma diferença importante em relação ao split anterior que merece atenção.

No histórico esparso original, as linhas estavam ordenadas por data e cada linha representava uma venda real. O corte 80/20 dividia dias do calendário de forma natural.

No painel completo, para cada data existem 1.084 linhas, com uma para cada combinação de rota × operadora. Quando ordenamos por data e cortamos em 80%, o corte ainda cai em uma data específica do calendário, mas agora cada "fatia" de data tem muito mais linhas do que antes.

O resultado prático é o mesmo: o modelo treina no passado e é testado no futuro. Mas vale ter consciência de que com apenas 25 dias de histórico e 1.084 combinações por dia, o conjunto de teste corresponde aos últimos  quase 5 dias do histórico; um período curto para avaliar sazonalidade. Conforme o histórico crescer, essa avaliação ficará mais robusta.

Uma limitação importante é que 25 dias é um período muito curto para um modelo com tantas combinações. O modelo vai aprender padrões de dia da semana e de rota, mas não vai conseguir capturar sazonalidade mensal ou anual, porque não há dados suficientes para isso. As previsões de 90 dias à frente vão extrapolar muito além do que o histórico cobre, o que torna os resultados **indicativos, não conclusivos**.

À medida que mais dados forem acumulando, vale retreinar o modelo mensalmente para incorporar padrões de médio e longo prazo.

In [ ]:
cols_nao_usar_painel = [
    "bilhetes", "data", "origem", "destino", "rota", "operadora"
]

feature_cols_painel = [c for c in df_painel.columns if c not in cols_nao_usar_painel]
target_col_painel   = "bilhetes"

# Ordenar cronologicamente
df_painel = df_painel.sort_values("data").reset_index(drop=True)

# Corte em 80/20
corte_painel = int(len(df_painel) * 0.8)

X_train_p = df_painel[feature_cols_painel].iloc[:corte_painel]
y_train_p = df_painel[target_col_painel].iloc[:corte_painel]

X_test_p  = df_painel[feature_cols_painel].iloc[corte_painel:]
y_test_p  = df_painel[target_col_painel].iloc[corte_painel:]

print(f"Treino: {len(X_train_p)} linhas")
print(f"Teste : {len(X_test_p)} linhas")
print(f"\nData de corte: {df_painel['data'].iloc[corte_painel].date()}")

Treino: 21687 linhas
Teste : 5422 linhas

Data de corte: 2025-11-21


#Treinamento do modelo corrigido

O novo modelo é treinado no painel completo com zeros. Os hiperparâmetros foram ajustados em relação ao modelo original por um motivo específico.

O painel completo tem uma característica diferente do histórico esparso: a maioria das linhas tem bilhetes = 0. Isso cria um dataset muito mais esparso e com maior risco de overfitting, pois o modelo pode aprender a "decorar" os poucos dias com venda de cada rota em vez de generalizar o padrão. Para compensar, dois ajustes foram feitos:

*   ``max_depth=15`` em vez de 20: árvores menos profundas generalizam melhor quando os dados têm muitos zeros. Profundidade 20 permitiria divisões muito específicas que provavelmente representam ruído, não padrão real.

*   ``min_samples_leaf=5`` em vez de 2: cada folha precisa ter pelo menos 5 exemplos. Com muitos zeros no dataset, folhas com 2 exemplos podem representar casos muito específicos; por exemplo, uma rota rara que vendeu exatamente 3 bilhetes em duas datas específicas. Exigir 5 exemplos **força o modelo** a encontrar padrões mais gerais.

Esses valores são um ponto de partida razoável. Se após avaliar o modelo os resultados ainda mostrarem overfitting (R^2 treino muito maior que R^2 teste),
>>**reduzir ``max_depth`` para 10 ou aumentar ``min_samples_leaf`` para 10 são os próximos ajustes a tentar.**

In [ ]:
# treinando novo Random Forest no painel completo
rf_painel = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=5,
    random_state=0,
    n_jobs=-1
)

rf_painel.fit(X_train_p, y_train_p)

print("Modelo treinado no painel completo.")
print(f"Features usadas: {feature_cols_painel}")

Modelo treinado no painel completo.
Features usadas: ['dia', 'mês', 'ano', 'dia_semana', 'semana_ano', 'mes_num', 'fim_semana', 'operadora_enc', 'origem_enc', 'destino_enc', 'rota_enc']


#Avaliação do modelo corrigido

In [ ]:
# Avaliar o modelo corrigido
resultados_painel = avaliar_modelo(
    rf_painel,
    X_train_p, y_train_p,
    X_test_p,  y_test_p,
    nome="Random Forest — Painel Completo"
)

===== Desempenho do Random Forest — Painel Completo =====
MAE Treino: 0.22
MAE Teste : 0.27

RMSE Treino: 0.55
RMSE Teste : 0.60

R² Treino: 0.804
R² Teste : 0.738


Como interpretar as métricas no painel completo
As métricas aqui precisam ser lidas com um olhar diferente do modelo anterior, por dois motivos:



*   **Motivo 1**: O MAE vai parecer baixo, mas cuidado com a ilusão dos zeros.
Com a maioria das linhas tendo bilhetes = 0, o modelo pode alcançar um MAE baixo simplesmente prevendo zero para tudo (e estaria certo na maioria dos casos). Isso é chamado de **zero-inflation bias**: o modelo aprende que "prever zero é uma estratégia segura" porque acerta a maioria das linhas, mesmo errando nas que realmente importam.

Por isso, além de olhar o MAE geral, vale checar separadamente o erro apenas nas linhas com bilhetes > 0:

In [ ]:
# Erro apenas nas linhas com venda real
mask_venda = y_test_p > 0
y_real_venda   = y_test_p[mask_venda]
y_prev_venda   = rf_painel.predict(X_test_p[mask_venda])

print(f"MAE apenas em linhas com venda: {mean_absolute_error(y_real_venda, y_prev_venda):.2f}")
print(f"R² apenas em linhas com venda : {r2_score(y_real_venda, y_prev_venda):.3f}")

MAE apenas em linhas com venda: 1.26
R² apenas em linhas com venda : 0.687


*   **Motivo 2**: O objetivo final não é prever bilhetes por linha com perfeição; é que a soma diária de todas as linhas fique próxima do histórico real.

Por isso, após avaliar as métricas por linha, o teste mais importante é comparar o total diário previsto com o total diário real no período de teste:

In [ ]:
# Comparar totais diários no período de teste
df_test_painel = df_painel.iloc[corte_painel:].copy()
df_test_painel["bilhetes_previstos"] = rf_painel.predict(X_test_p).clip(0).round().astype(int)

total_real    = df_test_painel.groupby("data")["bilhetes"].sum()
total_previsto = df_test_painel.groupby("data")["bilhetes_previstos"].sum()

comparacao = pd.DataFrame({
    "real"     : total_real,
    "previsto" : total_previsto,
    "erro_pct" : ((total_previsto - total_real) / total_real * 100).round(1)
})

print(comparacao)

            real  previsto  erro_pct
data                                
2025-11-21   253       168     -33.6
2025-11-22   231       176     -23.8
2025-11-23   261       184     -29.5
2025-11-24   198       178     -10.1
2025-11-25   183       169      -7.7


Se o erro percentual diário estiver consistentemente abaixo de 20%, o modelo já é útil para orientar decisões operacionais. Se ainda estiver alto, as causas mais prováveis são o histórico curto (25 dias) ou combinações muito raras que introduzem ruído.

#Previsão corrigida e comparação final

O pipeline de previsão futura segue a mesma estrutura da versão anterior:



*   construção do ``df_future``,
*   features de tempo,
*   máscara de válidos,
*   encoding e previsão (mas agora usando o modelo treinado no painel completo (rf_painel)), e,
*   os encoders correspondentes (``le_op2``, ``le_ori2``, etc.).

Com o modelo corrigido, a previsão diária agregada deve ficar em uma faixa muito mais próxima do histórico real (em torno de 200 a 300 bilhetes por dia, em vez dos quase 1.700 da versão anterior).

A comparação entre o último período do histórico e os primeiros dias da previsão é o teste de sanidade final: se os valores estiverem na mesma ordem de grandeza, o pipeline está funcionando corretamente.

**Alguns comportamentos esperados e normais:**


*   Fins de semana com volume diferente de dias úteis: o modelo aprendeu dia_semana e fim_semana, então vai naturalmente prever volumes distintos para sábados e domingos.

*   Variação entre dias: o modelo vai oscilar conforme o padrão aprendido de cada dia da semana e período do mês.

*   Previsões de longo prazo menos confiáveis: os primeiros 15 a 30 dias tendem a ser mais precisos. Além disso, o modelo está extrapolando padrões de apenas 25 dias de histórico; use as previsões de 60 a 90 dias como referência de tendência, não como número exato.

In [ ]:
# Construindo df_future no mesmo formato do painel
future_rows_p = []
for data_future in datas_futuras:
    for _, row in combos_reais.iterrows():
        future_rows_p.append({
            "data"      : data_future,
            "origem"    : row["origem"],
            "destino"   : row["destino"],
            "rota"      : row["rota"],
            "operadora" : row["operadora"]
        })

df_future_p = pd.DataFrame(future_rows_p)

# Features de tempo
df_future_p["dia"]        = df_future_p["data"].dt.day
df_future_p["mês"]        = df_future_p["data"].dt.month
df_future_p["ano"]        = df_future_p["data"].dt.year
df_future_p["dia_semana"] = df_future_p["data"].dt.weekday
df_future_p["semana_ano"] = df_future_p["data"].dt.isocalendar().week.astype(int)
df_future_p["mes_num"]    = df_future_p["mês"]
df_future_p["fim_semana"] = df_future_p["dia_semana"].isin([5, 6]).astype(int)

# Encoding com os novos encoders
mascara_p = (
    df_future_p["origem"].isin(le_ori2.classes_)    &
    df_future_p["destino"].isin(le_dest2.classes_)  &
    df_future_p["rota"].isin(le_rot2.classes_)      &
    df_future_p["operadora"].isin(le_op2.classes_)
)

df_future_p_valid = df_future_p[mascara_p].copy()

df_future_p_valid["operadora_enc"] = le_op2.transform(df_future_p_valid["operadora"])
df_future_p_valid["origem_enc"]    = le_ori2.transform(df_future_p_valid["origem"])
df_future_p_valid["destino_enc"]   = le_dest2.transform(df_future_p_valid["destino"])
df_future_p_valid["rota_enc"]      = le_rot2.transform(df_future_p_valid["rota"])

# Previsão
X_future_p = df_future_p_valid[feature_cols_painel]

df_future_p_valid["bilhetes_previstos"] = (
    rf_painel.predict(X_future_p)
    .clip(0)
    .round()
    .astype(int)
)

# Agregação por dia
previsao_corrigida = (
    df_future_p_valid
    .groupby("data")["bilhetes_previstos"]
    .sum()
    .reset_index()
)

# Comparação com histórico
print("Histórico (últimos 7 dias):")
print(historico_por_dia.tail(7))
print("\nPrevisão corrigida (primeiros 7 dias):")
print(previsao_corrigida.head(7))

Histórico (últimos 7 dias):
         data  bilhetes_reais
18 2025-11-19             273
19 2025-11-20             217
20 2025-11-21             253
21 2025-11-22             231
22 2025-11-23             261
23 2025-11-24             198
24 2025-11-25             183

Previsão corrigida (primeiros 7 dias):
        data  bilhetes_previstos
0 2025-11-26                 160
1 2025-11-27                 165
2 2025-11-28                 168
3 2025-11-29                 176
4 2025-11-30                 184
5 2025-12-01                 169
6 2025-12-02                 165


## Previsão por rota

Além do total diário, o modelo agora permite consultar a previsão desagregada por rota, o que era o objetivo original. Para ver as rotas com maior volume previsto em um dia específico:

In [ ]:
# Previsão por rota para um dia específico
dia_consulta = previsao_corrigida["data"].iloc[0]  # primeiro dia previsto

top_rotas_previstas = (
    df_future_p_valid[df_future_p_valid["data"] == dia_consulta]
    [["rota", "operadora", "bilhetes_previstos"]]
    .sort_values("bilhetes_previstos", ascending=False)
    .head(100)
)

#print(f"Top 10 rotas previstas para {dia_consulta.date()}:")
#print(top_rotas_previstas)
top_rotas_previstas

,rota,operadora,bilhetes_previstos
20,PONTA GROSSA - CURITIBA,PRINCESA DOS CAMPOS,22
102,CURITIBA - PONTA GROSSA,PRINCESA DOS CAMPOS,16
79,FLORIANOPOLIS - PORTO ALEGRE,BRASIL SUL,13
74,PORTO ALEGRE - FLORIANOPOLIS,BRASIL SUL,12
95,CURITIBA - REGISTRO,PRINCESA DOS CAMPOS,7
...,...,...,...
816,CAMPINAS - JAGUARIUNA,RAPIDO FENIX,0
712,CASCAVEL - SANTO ANTONIO DO SUDOESTE,LOPESUL,0
713,TANGARA DA SERRA - CUIABA,GENESIS BUS,0
714,PONTA GROSSA - PATO BRANCO,PRINCESA DOS CAMPOS,0
